In [4]:
import scrapy
import json

In [3]:
class ProductDetailItem(scrapy.Item):
    url = scrapy.Field()
    title = scrapy.Field()
    price = scrapy.Field()
    location = scrapy.Field()
    description = scrapy.Field()
    crawled_at = scrapy.Field()
    brand = scrapy.Field()
    line_product = scrapy.Field()
    device_condition = scrapy.Field()
    screen_size = scrapy.Field()
    cpu = scrapy.Field()
    ram = scrapy.Field()
    graphic_card = scrapy.Field()
    hard_drive = scrapy.Field()
    hard_drive_type = scrapy.Field()

In [7]:
class FullSipder(scrapy.Spider):
    
    name = "test"
    base_api = "https://gateway.chotot.com/v1/public/ad-listing?limit=50&protection_entitlement=true&event_source=ct_home_highlight_category&cg=5030&region_v2=12000&st=s,k&key_param_included=true&video_count_included=false&o={offset}"
    
    MAX_ITEMS = 2000
    PAGE_SIZE = 50
    
    def start_request(self):
        return scrapy.Request(
            url=self.base_api.format(offset=0),
            callback=self.parse,
            meta={"offset": 0}
        )
        
    def parse(self, response):
        data = json.loads(response.text)
        ads = data.get("ads", [])
        total = data.get("total", 0)
        current_offset = response.meta["offset"]
        self.logger.info(f"Scraping from {current_offset} / Total ~{total}")
        
        for ad in ads:
            item = ProductDetailItem()
            item["url"] = f"https://www.chotot.com/{ad.get('list_id')}.htm"
            item["title"] = ad.get("subject", "").strip()
            item["price"] = str(ad.get("price", ""))
            item["description"] = ad.get("body", "").strip()
            
            # Bóc tách thông số kỹ thuật từ parameters
            params = {p.get("id"): p.get("value") for p in ad.get("parameters", [])}
            item["brand"] = params.get("brand", "")
            item["cpu"] = params.get("laptop_cpu", "")
            item["ram"] = params.get("ram", "") 
            
            display_json(item)
    
        next_offset = current_offset + self.PAGE_SIZE
        
        if ads and next_offset < total and next_offset < self.MAX_ITEMS:
            return scrapy.Request(
                url=self.base_api.format(offset=next_offset),
                callback=self.parse,
                meta={"offset": next_offset}
            )  
        else:
            self.logger.info("ĐÃ QUÉT HẾT BÀI ĐĂNG HOẶC ĐẠT GIỚI HẠN!")

In [8]:
item = FullSipder()

In [10]:
print(item)

<FullSipder 'test' at 0x1ecd7247710>
